In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
PROJECT_DIR = '/content/drive/MyDrive/fake-job-detector'
%cd $PROJECT_DIR

/content/drive/MyDrive/fake-job-detector


In [3]:
%%writefile src/preprocess.py
import re
import pandas as pd
from src.config import TEXT_COLS


def build_combined_text(df):
    return df[TEXT_COLS].fillna('').agg(' '.join, axis=1)


def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^a-z0-9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

Overwriting src/preprocess.py


In [4]:
from src.preprocess import clean_text

clean_text('<p>Earn $500/day!!  No   Experience NEEDED.</p>')

'earn 500 day no experience needed'

In [5]:
import pandas as pd
from src.config import DATA_PATH, TARGET_COL
from src.preprocess import build_combined_text, clean_text

df_raw = pd.read_csv(DATA_PATH)

df_clean = pd.DataFrame()
df_clean['combined_text'] = build_combined_text(df_raw)
df_clean['clean_text'] = df_clean['combined_text'].apply(clean_text)
df_clean[TARGET_COL] = df_raw[TARGET_COL]

df_clean.head()

,combined_text,clean_text,fraudulent
0,"Marketing Intern We're Food52, and we've creat...",marketing intern we re food52 and we ve create...,0
1,Customer Service - Cloud Video Production 90 S...,customer service cloud video production 90 sec...,0
2,Commissioning Machinery Assistant (CMA) Valor ...,commissioning machinery assistant cma valor se...,0
3,Account Executive - Washington DC Our passion ...,account executive washington dc our passion fo...,0
4,Bill Review Manager SpotSource Solutions LLC i...,bill review manager spotsource solutions llc i...,0


In [6]:
(df_clean['clean_text'] == '').sum()

np.int64(0)

In [7]:
from sklearn.model_selection import train_test_split
from src.config import TEST_SIZE, RANDOM_STATE

X = df_clean['clean_text']
y = df_clean[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(X_train.shape, X_test.shape)

(14304,) (3576,)


In [8]:
y_train.value_counts(normalize=True)

,proportion
fraudulent,
0,0.951552
1,0.048448


In [9]:
df_clean.to_csv('data/df_clean.csv', index=False)

In [10]:
import sklearn
print(sklearn.__version__)
!python --version

1.6.1
Python 3.13.16
